# Caracal Shadow Eval · TPU paralelo ao treino GPU

Roda em TPU enquanto outros founders treinam em GPU. Loop:
1. Poll Kaggle Datasets dos founders
2. Quando achar `caracal-base-3b-sNN`, pull
3. Roda probe + CyberGym slice-50
4. Posta resultado em Kaggle Dataset proprio
5. Repete pra 5 sessoes

**Antes de Run All:**
1. Settings -> Accelerator -> TPU VM v3-8 (ou v5e-8)
2. Settings -> Internet -> ON
3. Settings -> Persistence -> Variables and Files
4. Edita SHADOW_OUTPUT_SLUG abaixo

In [ ]:
SHADOW_OUTPUT_SLUG = "pedroafonso2/caracal-shadow-eval"
POLL_INTERVAL_SEC = 1800
MAX_WAIT_HOURS = 60
print(f"Shadow output: {SHADOW_OUTPUT_SLUG} | poll={POLL_INTERVAL_SEC}s")

In [ ]:
!pip install -q 'transformers>=4.46.0' 'peft>=0.13.0' 'datasets>=3.0.0' kaggle

In [ ]:
import os, subprocess

if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'dev', 'https://github.com/iterate-labs-ai/caracal-1.git', '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')
rev = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
print(f'cloned, HEAD={rev}', flush=True)

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    print(f'XLA device: {device}')
except ImportError:
    print('torch_xla nao instalado, vai cair em CPU')

In [ ]:
cmd = f'python -u eval/shadow_loop.py --shadow-output {SHADOW_OUTPUT_SLUG} --poll-interval {POLL_INTERVAL_SEC} --max-wait-hours {MAX_WAIT_HOURS}'
print('Running:', cmd, flush=True)
!{cmd}

## Resultados

Cada sessao publicada em `SHADOW_OUTPUT_SLUG` com:
- mean_ppl (probe set)
- cwe_hit_rate (51 probes)
- pass_at_1 (CyberGym slice-50)

Founders veem delta entre sessoes consecutivas em real-time.